In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# =====================================================================
# 1. LOAD THE TARGET DATASETS
# =====================================================================
print("Loading disparate cohort schemas...")
df4 = pd.read_csv('training_dataset.csv')     # MIMIC-IV Data File
df3 = pd.read_csv('MIMIC_III_CARDIAC_ML_READY.csv')   # MIMIC-III Data File

# Standardize binary outcome variables
y4 = df4['hospital_expire_flag'].astype(int)
y3 = df3['hospital_expire_flag'].astype(int)

hadm_id_4 = df4['hadm_id']
hadm_id_3 = df3['hadm_id']

# =====================================================================
# 2. ISOLATE DEMOGRAPHIC & OPERATIONAL BASELINES
# =====================================================================
# Extract structural features present across both source schemas
shared_features = ['anchor_age', 'visit_procedure_count', 'is_male']

X4_shared = df4[shared_features].copy()
X3_shared = df3[shared_features].copy()

# =====================================================================
# 3. INTERSECT ICD TAXONOMY VECTORS
# =====================================================================
# Generate binary structural keys from the parsed text descriptions
X4_dx = pd.get_dummies(df4['icd_description'], prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))
X3_dx = pd.get_dummies(df3['icd_description'], prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Intersect column layouts ensuring structural feature alignment
X4_dx_aligned, X3_dx_aligned = X4_dx.align(X3_dx, join='inner', axis=1, fill_value=0)

# Merge continuous metrics and structural categorical dimensions
X4_final = pd.concat([X4_shared, X4_dx_aligned], axis=1)
X3_final = pd.concat([X3_shared, X3_dx_aligned], axis=1)

print(f"Matrix intersection finalized: {X4_final.shape[1]} operational features aligned.")

# Target best parameter configurations discovered during the cross-validation optimization sweep
BEST_C = 0.0081
BEST_L1 = 0.50

def evaluate_patient_level(probs, true_y, hadm_ids, name):
    summary_df = pd.DataFrame({'hadm_id': hadm_ids, 'y_true': true_y, 'y_prob': probs})
    patient_res = summary_df.groupby('hadm_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)
    print(f"\n=======================================================\n  {name} EXTERNAL PATIENT-LEVEL SUMMARY\n=======================================================")
    print(classification_report(patient_res['y_true'], preds))
    print(f"{name} Patient External ROC-AUC: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")


# =====================================================================
# EXPERIMENT DIRECTION A: TRAIN MIMIC-IV ➔ TEST MIMIC-III
# =====================================================================
print("\n[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...")

X_train_A, y_train_A = X4_final.copy(), y4.copy()
X_test_A, y_test_A = X3_final.copy(), y3.copy()

# Prevent scaling leakage across cohorts
scaler_A = StandardScaler()
X_train_A[['anchor_age', 'visit_procedure_count']] = scaler_A.fit_transform(X_train_A[['anchor_age', 'visit_procedure_count']])
X_test_A[['anchor_age', 'visit_procedure_count']] = scaler_A.transform(X_test_A[['anchor_age', 'visit_procedure_count']])

model_A = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_A.fit(X_train_A, y_train_A)

probs_A = model_A.predict_proba(X_test_A)[:, 1]
evaluate_patient_level(probs_A, y_test_A, hadm_id_3, "MIMIC-IV TO MIMIC-III")


# =====================================================================
# EXPERIMENT DIRECTION B: TRAIN MIMIC-III ➔ TEST MIMIC-IV
# =====================================================================
print("\n[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...")

X_train_B, y_train_B = X3_final.copy(), y3.copy()
X_test_B, y_test_B = X4_final.copy(), y4.copy()

# Isolate feature statistics specifically inside MIMIC-III bounds
scaler_B = StandardScaler()
X_train_B[['anchor_age', 'visit_procedure_count']] = scaler_B.fit_transform(X_train_B[['anchor_age', 'visit_procedure_count']])
X_test_B[['anchor_age', 'visit_procedure_count']] = scaler_B.transform(X_test_B[['anchor_age', 'visit_procedure_count']])

model_B = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_B.fit(X_train_B, y_train_B)

probs_B = model_B.predict_proba(X_test_B)[:, 1]
evaluate_patient_level(probs_B, y_test_B, hadm_id_4, "MIMIC-III TO MIMIC-IV")

Loading disparate cohort schemas...
Matrix intersection finalized: 182 operational features aligned.

[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...

  MIMIC-IV TO MIMIC-III EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.89      0.69      0.78     31451
         1.0       0.20      0.48      0.28      4945

    accuracy                           0.67     36396
   macro avg       0.55      0.59      0.53     36396
weighted avg       0.80      0.67      0.71     36396

MIMIC-IV TO MIMIC-III Patient External ROC-AUC: 0.6288

[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...

  MIMIC-III TO MIMIC-IV EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.99      0.68      0.81      3366
         1.0       0.06      0.76      0.11        93

    accuracy                           0.68      3459
   macro av

In [3]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# =====================================================================
# 1. LOAD THE TARGET DATASETS
# =====================================================================
print("Loading disparate cohort schemas...")
df4 = pd.read_csv('training_dataset.csv')     # MIMIC-IV Data File
df3 = pd.read_csv('MIMIC_III_CARDIAC_ML_READY.csv')   # MIMIC-III Data File

# Standardize binary outcome variables
y4 = df4['hospital_expire_flag'].astype(int)
y3 = df3['hospital_expire_flag'].astype(int)

hadm_id_4 = df4['hadm_id']
hadm_id_3 = df3['hadm_id']

# =====================================================================
# 2. ISOLATE DEMOGRAPHIC & OPERATIONAL BASELINES
# =====================================================================
shared_features = ['anchor_age', 'visit_procedure_count', 'is_male']

# FIX: Added .fillna(0) here to protect continuous features from having NaNs
X4_shared = df4[shared_features].copy().fillna(0)
X3_shared = df3[shared_features].copy().fillna(0)

# =====================================================================
# 3. INTERSECT ICD TAXONOMY VECTORS
# =====================================================================
# Generate binary structural keys from the parsed text descriptions
# Added dummy_na=False to prevent explicit NaN columns from forming
X4_dx = pd.get_dummies(df4['icd_description'].fillna('missing'), prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))
X3_dx = pd.get_dummies(df3['icd_description'].fillna('missing'), prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Intersect column layouts ensuring structural feature alignment
X4_dx_aligned, X3_dx_aligned = X4_dx.align(X3_dx, join='inner', axis=1, fill_value=0)

# FIX: Align the indices of our shared features and text features before concat
X4_shared.index = X4_dx_aligned.index
X3_shared.index = X3_dx_aligned.index

# Merge continuous metrics and structural categorical dimensions
# FIX: Added .fillna(0) at the end of concat to wipe out any remaining structural NaNs
X4_final = pd.concat([X4_shared, X4_dx_aligned], axis=1).fillna(0)
X3_final = pd.concat([X3_shared, X3_dx_aligned], axis=1).fillna(0)

print(f"Matrix intersection finalized: {X4_final.shape[1]} operational features aligned.")

# Target best parameter configurations discovered during the cross-validation optimization sweep
BEST_C = 0.0081
BEST_L1 = 0.50

def evaluate_patient_level(probs, true_y, hadm_ids, name):
    # FIX: Explicitly reset indices to prevent alignment issues during evaluation dataframe creation
    summary_df = pd.DataFrame({
        'hadm_id': hadm_ids.reset_index(drop=True),
        'y_true': true_y.reset_index(drop=True),
        'y_prob': probs
    })
    patient_res = summary_df.groupby('hadm_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)
    print(f"\n=======================================================\n  {name} EXTERNAL PATIENT-LEVEL SUMMARY\n=======================================================")
    print(classification_report(patient_res['y_true'], preds))
    print(f"{name} Patient External ROC-AUC: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")


# =====================================================================
# EXPERIMENT DIRECTION A: TRAIN MIMIC-IV ➔ TEST MIMIC-III
# =====================================================================
print("\n[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...")

X_train_A, y_train_A = X4_final.copy(), y4.copy()
X_test_A, y_test_A = X3_final.copy(), y3.copy()

# Prevent scaling leakage across cohorts
scaler_A = StandardScaler()
X_train_A[['anchor_age', 'visit_procedure_count']] = scaler_A.fit_transform(X_train_A[['anchor_age', 'visit_procedure_count']])
X_test_A[['anchor_age', 'visit_procedure_count']] = scaler_A.transform(X_test_A[['anchor_age', 'visit_procedure_count']])

model_A = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_A.fit(X_train_A, y_train_A)

probs_A = model_A.predict_proba(X_test_A)[:, 1]
evaluate_patient_level(probs_A, y_test_A, hadm_id_3, "MIMIC-IV TO MIMIC-III")


# =====================================================================
# EXPERIMENT DIRECTION B: TRAIN MIMIC-III ➔ TEST MIMIC-IV
# =====================================================================
print("\n[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...")

X_train_B, y_train_B = X3_final.copy(), y3.copy()
X_test_B, y_test_B = X4_final.copy(), y4.copy()

# Isolate feature statistics specifically inside MIMIC-III bounds
scaler_B = StandardScaler()
X_train_B[['anchor_age', 'visit_procedure_count']] = scaler_B.fit_transform(X_train_B[['anchor_age', 'visit_procedure_count']])
X_test_B[['anchor_age', 'visit_procedure_count']] = scaler_B.transform(X_test_B[['anchor_age', 'visit_procedure_count']])

model_B = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_B.fit(X_train_B, y_train_B)

probs_B = model_B.predict_proba(X_test_B)[:, 1]
evaluate_patient_level(probs_B, y_test_B, hadm_id_4, "MIMIC-III TO MIMIC-IV")

Loading disparate cohort schemas...
Matrix intersection finalized: 182 operational features aligned.

[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...

  MIMIC-IV TO MIMIC-III EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.89      0.69      0.78     31451
         1.0       0.20      0.48      0.28      4945

    accuracy                           0.67     36396
   macro avg       0.55      0.59      0.53     36396
weighted avg       0.80      0.67      0.71     36396

MIMIC-IV TO MIMIC-III Patient External ROC-AUC: 0.6288

[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...

  MIMIC-III TO MIMIC-IV EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.99      0.68      0.81      3366
         1.0       0.06      0.76      0.11        93

    accuracy                           0.68      3459
   macro av

In [7]:
# =====================================================================
# FIXED FEATURE ALIGNMENT SECTION
# =====================================================================

# 1. Map and standardize your continuous vital signs explicitly if they have different names
# (Example: If MIMIC-3 uses a different naming convention, rename them here to match MIMIC-IV)
# df3 = df3.rename(columns={'raw_bp_systolic': 'vital_mean_systolic_bp', 'raw_hr': 'vital_mean_heart_rate'})

# 2. Harmonize text by cleaning the ICD descriptions BEFORE running get_dummies
# This strips punctuation and forces lowercase so "Stage III" and "stage 3" can align naturally
df4['clean_dx'] = df4['icd_description'].astype(str).str.lower().str.replace(r'[^a-z0-9]', '_', regex=True)
df3['clean_dx'] = df3['icd_description'].astype(str).str.lower().str.replace(r'[^a-z0-9]', '_', regex=True)

# 3. Now run your one-hot encoding on the harmonized text columns
X4_dx = pd.get_dummies(df4['clean_dx'], prefix='Dx')
X3_dx = pd.get_dummies(df3['clean_dx'], prefix='Dx')

# 4. Include your demographic foundations safely
shared_features = ['anchor_age', 'visit_procedure_count', 'is_male']
# Add vital signs to this list once their column names are identical in df3 and df4!
# shared_features += ['vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp']

X4_shared = df4[shared_features].copy()
X3_shared = df3[shared_features].copy()

# Intersect the columns cleanly
X4_dx_aligned, X3_dx_aligned = X4_dx.align(X3_dx, join='inner', axis=1, fill_value=0)

X4_final = pd.concat([X4_shared, X4_dx_aligned], axis=1)
X3_final = pd.concat([X3_shared, X3_dx_aligned], axis=1)

print(f"Fixed alignment! Matrix space now contains {X4_final.shape[1]} highly aligned clinical features.")

Fixed alignment! Matrix space now contains 182 highly aligned clinical features.


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# =====================================================================
# 1. LOAD THE TARGET DATASETS & DEFINE TARGET VECTORS CORRECTLY
# =====================================================================
print("Loading disparate cohort schemas...")
df4 = pd.read_csv('training_dataset.csv')     # MIMIC-IV Data File
df3 = pd.read_excel('MIMIC_III_CARDIAC_ML_READY.xlsx')   # MIMIC-III Data File

# FIX: Natively map and isolate the separate survival target targets right here
y4 = df4['hospital_expire_flag'].astype(int)
y3 = df3['hospital_expire_flag'].astype(int)  # MIMIC-III standard label name

hadm_id_4 = df4['hadm_id']
hadm_id_3 = df3['hadm_id']

# =====================================================================
# FIXED FEATURE ALIGNMENT SECTION
# =====================================================================

# 1. Map and standardize your continuous vital signs explicitly if they have different names
# (Uncomment and edit below if your MIMIC-III column names differ)
# df3 = df3.rename(columns={'raw_bp_systolic': 'vital_mean_systolic_bp'})

# 2. Harmonize text by cleaning the ICD descriptions BEFORE running get_dummies
# FIX: Added .fillna('') to protect against blank or missing entries in the text columns
df4['clean_dx'] = df4['icd_description'].astype(str).fillna('').str.lower().str.replace(r'[^a-z0-9]', '_', regex=True)
df3['clean_dx'] = df3['icd_description'].astype(str).fillna('').str.lower().str.replace(r'[^a-z0-9]', '_', regex=True)

# 3. Now run your one-hot encoding on the harmonized text columns
X4_dx = pd.get_dummies(df4['clean_dx'], prefix='Dx')
X3_dx = pd.get_dummies(df3['clean_dx'], prefix='Dx')

# 4. Include your demographic foundations safely
shared_features = ['anchor_age', 'visit_procedure_count', 'is_male']
# Add vital signs to this list once their column names are identical in df3 and df4!
# shared_features += ['vital_mean_heart_rate', 'vital_mean_systolic_bp', 'vital_mean_diastolic_bp']

X4_shared = df4[shared_features].copy()
X3_shared = df3[shared_features].copy()

# Intersect the columns cleanly
X4_dx_aligned, X3_dx_aligned = X4_dx.align(X3_dx, join='inner', axis=1, fill_value=0)

X4_final = pd.concat([X4_shared, X4_dx_aligned], axis=1)
X3_final = pd.concat([X3_shared, X3_dx_aligned], axis=1)

print(f"Fixed alignment! Matrix space now contains {X4_final.shape[1]} highly aligned clinical features.")

# Target best parameter configurations discovered during the cross-validation optimization sweep
BEST_C = 0.0081
BEST_L1 = 0.50

def evaluate_patient_level(probs, true_y, hadm_ids, name):
    # FIX: Explicitly resets indices to avoid mismatch alignment warnings inside the evaluation tracking frame
    summary_df = pd.DataFrame({
        'hadm_id': hadm_ids.reset_index(drop=True),
        'y_true': true_y.reset_index(drop=True),
        'y_prob': probs
    })
    patient_res = summary_df.groupby('hadm_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)
    print(f"\n=======================================================\n  {name} EXTERNAL PATIENT-LEVEL SUMMARY\n=======================================================")
    print(classification_report(patient_res['y_true'], preds))
    print(f"{name} Patient External ROC-AUC: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")


# =====================================================================
# EXPERIMENT DIRECTION A: TRAIN MIMIC-IV ➔ TEST MIMIC-III
# =====================================================================
print("\n[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...")

X_train_A, y_train_A = X4_final.copy(), y4.copy()
X_test_A, y_test_A = X3_final.copy(), y3.copy()

# Prevent scaling leakage across cohorts
scaler_A = StandardScaler()
X_train_A[['anchor_age', 'visit_procedure_count']] = scaler_A.fit_transform(X_train_A[['anchor_age', 'visit_procedure_count']])
X_test_A[['anchor_age', 'visit_procedure_count']] = scaler_A.transform(X_test_A[['anchor_age', 'visit_procedure_count']])

model_A = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_A.fit(X_train_A, y_train_A)

probs_A = model_A.predict_proba(X_test_A)[:, 1]
evaluate_patient_level(probs_A, y_test_A, hadm_id_3, "MIMIC-IV TO MIMIC-III")


# =====================================================================
# EXPERIMENT DIRECTION B: TRAIN MIMIC-III ➔ TEST MIMIC-IV
# =====================================================================
print("\n[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...")

X_train_B, y_train_B = X3_final.copy(), y3.copy()
X_test_B, y_test_B = X4_final.copy(), y4.copy()

# Isolate feature statistics specifically inside MIMIC-III bounds
scaler_B = StandardScaler()
X_train_B[['anchor_age', 'visit_procedure_count']] = scaler_B.fit_transform(X_train_B[['anchor_age', 'visit_procedure_count']])
X_test_B[['anchor_age', 'visit_procedure_count']] = scaler_B.transform(X_test_B[['anchor_age', 'visit_procedure_count']])

model_B = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_B.fit(X_train_B, y_train_B)

probs_B = model_B.predict_proba(X_test_B)[:, 1]
evaluate_patient_level(probs_B, y_test_B, hadm_id_4, "MIMIC-III TO MIMIC-IV")

Loading disparate cohort schemas...
Fixed alignment! Matrix space now contains 182 highly aligned clinical features.

[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...

  MIMIC-IV TO MIMIC-III EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.89      0.69      0.78     31451
         1.0       0.20      0.48      0.28      4945

    accuracy                           0.67     36396
   macro avg       0.55      0.59      0.53     36396
weighted avg       0.80      0.67      0.71     36396

MIMIC-IV TO MIMIC-III Patient External ROC-AUC: 0.6288

[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...

  MIMIC-III TO MIMIC-IV EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.99      0.68      0.81      3366
         1.0       0.06      0.76      0.11        93

    accuracy                           0.68      

In [8]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# =====================================================================
# 1. LOAD THE TARGET DATASETS
# =====================================================================
print("Loading disparate cohort schemas...")
df4 = pd.read_csv('training_dataset.csv')     # MIMIC-IV Data File
df3 = pd.read_csv('MIMIC_III_CARDIAC_ML_READY.csv')   # MIMIC-III Data File

# Standardize binary outcome variables
y4 = df4['hospital_expire_flag'].astype(int)
y3 = df3['hospital_expire_flag'].astype(int)

hadm_id_4 = df4['hadm_id']
hadm_id_3 = df3['hadm_id']

# =====================================================================
# 2. ISOLATE ALL IDENTIFIED COMMON FEATURES
# =====================================================================
# Group 1: Continuous columns that require standard scaling
continuous_features = [
    'anchor_age', 'visit_procedure_count',
    'vital_mean_heart_rate', 'vital_mean_systolic_bp',
    'vital_mean_diastolic_bp', 'vital_mean_spo2', 'vital_mean_temp'
]

# Group 2: Binary/Frequency features that remain unscaled
binary_text_features = [
    'is_male', 'report_left', 'title_artery',
    'title_coronary', 'title_coronary_artery', 'title_heart'
]

all_shared_columns = continuous_features + binary_text_features

# Isolate columns and fill missing entries with 0 (or median for continuous vitals)
X4_shared = df4[all_shared_columns].copy()
X3_shared = df3[all_shared_columns].copy()

# Fill missing vitals/demographics safely to prevent solver calculation crashes
for col in continuous_features:
    X4_shared[col] = X4_shared[col].fillna(X4_shared[col].median())
    X3_shared[col] = X3_shared[col].fillna(X3_shared[col].median())

X4_shared = X4_shared.fillna(0)
X3_shared = X3_shared.fillna(0)

# =====================================================================
# 3. INTERSECT ICD TAXONOMY VECTORS
# =====================================================================
# Generate binary structural keys from the parsed text descriptions
X4_dx = pd.get_dummies(df4['icd_description'].fillna('missing'), prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))
X3_dx = pd.get_dummies(df3['icd_description'].fillna('missing'), prefix='Dx').rename(columns=lambda x: re.sub(r'[^A-Za-z0-9_]+', '', x))

# Intersect column layouts ensuring structural feature alignment
X4_dx_aligned, X3_dx_aligned = X4_dx.align(X3_dx, join='inner', axis=1, fill_value=0)

# Align the indices of our shared features and text features before concat
X4_shared.index = X4_dx_aligned.index
X3_shared.index = X3_dx_aligned.index

# Merge all continuous vitals, demographic settings, and structural text categoricals
X4_final = pd.concat([X4_shared, X4_dx_aligned], axis=1).fillna(0)
X3_final = pd.concat([X3_shared, X3_dx_aligned], axis=1).fillna(0)

print(f"Matrix intersection finalized: {X4_final.shape[1]} operational features aligned.")

# Target best parameter configurations discovered during the cross-validation optimization sweep
BEST_C = 0.0081
BEST_L1 = 0.50

def evaluate_patient_level(probs, true_y, hadm_ids, name):
    summary_df = pd.DataFrame({
        'hadm_id': hadm_ids.reset_index(drop=True),
        'y_true': true_y.reset_index(drop=True),
        'y_prob': probs
    })
    patient_res = summary_df.groupby('hadm_id').mean()
    preds = (patient_res['y_prob'] >= 0.5).astype(int)
    print(f"\n=======================================================\n  {name} EXTERNAL PATIENT-LEVEL SUMMARY\n=======================================================")
    print(classification_report(patient_res['y_true'], preds))
    print(f"{name} Patient External ROC-AUC: {roc_auc_score(patient_res['y_true'], patient_res['y_prob']):.4f}")


# =====================================================================
# EXPERIMENT DIRECTION A: TRAIN MIMIC-IV ➔ TEST MIMIC-III
# =====================================================================
print("\n[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...")

X_train_A, y_train_A = X4_final.copy(), y4.copy()
X_test_A, y_test_A = X3_final.copy(), y3.copy()

# Prevent scaling leakage across cohorts by grouping all continuous columns together
scaler_A = StandardScaler()
X_train_A[continuous_features] = scaler_A.fit_transform(X_train_A[continuous_features])
X_test_A[continuous_features] = scaler_A.transform(X_test_A[continuous_features])

model_A = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_A.fit(X_train_A, y_train_A)

probs_A = model_A.predict_proba(X_test_A)[:, 1]
evaluate_patient_level(probs_A, y_test_A, hadm_id_3, "MIMIC-IV TO MIMIC-III")


# =====================================================================
# EXPERIMENT DIRECTION B: TRAIN MIMIC-III ➔ TEST MIMIC-IV
# =====================================================================
print("\n[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...")

X_train_B, y_train_B = X3_final.copy(), y3.copy()
X_test_B, y_test_B = X4_final.copy(), y4.copy()

# Isolate feature statistics specifically inside MIMIC-III bounds for all vitals
scaler_B = StandardScaler()
X_train_B[continuous_features] = scaler_B.fit_transform(X_train_B[continuous_features])
X_test_B[continuous_features] = scaler_B.transform(X_test_B[continuous_features])

model_B = LogisticRegression(
    penalty='elasticnet', solver='saga', C=BEST_C, l1_ratio=BEST_L1,
    class_weight='balanced', max_iter=250, random_state=42
)
model_B.fit(X_train_B, y_train_B)

probs_B = model_B.predict_proba(X_test_B)[:, 1]
evaluate_patient_level(probs_B, y_test_B, hadm_id_4, "MIMIC-III TO MIMIC-IV")

Loading disparate cohort schemas...
Matrix intersection finalized: 192 operational features aligned.

[Run A] Training on MIMIC-IV (Source) ➔ Externally Validating on MIMIC-III (Target)...

  MIMIC-IV TO MIMIC-III EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.92      0.75      0.83     31451
         1.0       0.27      0.61      0.38      4945

    accuracy                           0.73     36396
   macro avg       0.60      0.68      0.60     36396
weighted avg       0.84      0.73      0.76     36396

MIMIC-IV TO MIMIC-III Patient External ROC-AUC: 0.7332

[Run B] Training on MIMIC-III (Source) ➔ Externally Validating on MIMIC-IV (Target)...

  MIMIC-III TO MIMIC-IV EXTERNAL PATIENT-LEVEL SUMMARY
              precision    recall  f1-score   support

         0.0       0.99      0.83      0.90      3366
         1.0       0.11      0.76      0.19        93

    accuracy                           0.83      3459
   macro av

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [ ]:
# mainone is above

In [10]:
import pandas as pd

# Assuming df_mimic3 and df_mimic4 are your existing DataFrames

# 1. Get the lists of unique NLP/exclusive columns for both
# (Excluding 'gender' and 'is_deceased' which are demographic)
m3_exclusive = [
    'report_10', 'report_100', 'report_11', 'report_12', 'report_13', 'report_14', 'report_15',
    'report_18', 'report_20', 'report_24', 'report_25', 'report_30', 'report_admission', 'report_artery',
    'report_blood', 'report_chest', 'report_ct', 'report_daily', 'report_daily_daily', 'report_date',
    'report_day', 'report_discharge', 'report_disease', 'report_dr', 'report_given', 'report_history',
    'report_home', 'report_hospital', 'report_hospital1', 'report_hours', 'report_medications', 'report_mg',
    'report_mg_tablet', 'report_normal', 'report_pain', 'report_patient', 'report_po', 'report_po_daily',
    'report_pt', 'report_right', 'report_sig', 'report_sig_tablet', 'report_status', 'report_tablet',
    'report_tablet_po', 'report_tablet_sig', 'report_time', 'report_times', 'report_valve', 'title_acute',
    'title_atherosclerosis', 'title_atherosclerosis_native', 'title_care', 'title_cerebral', 'title_chronic',
    'title_coronary_atherosclerosis', 'title_episode', 'title_episode_care', 'title_failure', 'title_heart_failure',
    'title_hemorrhage', 'title_infarction', 'title_infarction_initial', 'title_initial', 'title_initial_episode',
    'title_malignant', 'title_mention', 'title_native', 'title_native_coronary', 'title_septicemia',
    'title_subendocardial', 'title_subendocardial_infarction', 'title_unspecified', 'title_unspecified_septicemia',
    'title_valve'
]

m4_exclusive = [
    'report_1st', 'report_1st_degree', 'report_abnormal', 'report_abnormal_ecg', 'report_age',
    'report_age_undetermined', 'report_anterior', 'report_atrial', 'report_atrial_fibrillation', 'report_axis',
    'report_axis_deviation', 'report_block', 'report_borderline', 'report_changes', 'report_changes_myocardial',
    'report_changes_nonspecific', 'report_degree', 'report_degree_block', 'report_deviation', 'report_ecg',
    'report_ecg_sinus', 'report_extensive', 'report_fibrillation', 'report_hypertrophy', 'report_infarct',
    'report_infarct_age', 'report_inferior', 'report_inferior_infarct', 'report_inferior_lateral', 'report_ischemia',
    'report_lateral', 'report_lateral_st', 'report_leads', 'report_myocardial', 'report_myocardial_ischemia',
    'report_nonspecific', 'report_possible', 'report_response', 'report_rhythm', 'report_sinus', 'report_sinus_rhythm',
    'report_st', 'report_st_changes', 'report_undetermined', 'report_ventricular', 'report_ventricular_hypertrophy',
    'report_ventricular_response', 'report_wave', 'report_wave_changes', 'title_angioplasty', 'title_approach',
    'title_arteriography', 'title_arteriography_using', 'title_cardiac', 'title_cardiac_catheterization',
    'title_catheterization', 'title_catheters', 'title_coronary_arteriography', 'title_drug', 'title_drug_eluting',
    'title_eluting', 'title_heart_cardiac', 'title_insertion', 'title_left', 'title_left_heart', 'title_percutaneous',
    'title_percutaneous_approach', 'title_procedure', 'title_right', 'title_single', 'title_stent',
    'title_transluminal', 'title_transluminal_coronary', 'title_using', 'title_using_catheters'
]

# 2. Add missing columns to MIMIC-III with 0
for col in m4_exclusive:
    if col not in df3.columns:
        df3[col] = 0

# 3. Add missing columns to MIMIC-IV with 0
for col in m4_exclusive:
    if col not in df4.columns:
        df4[col] = 0

# 4. Standardize demographics before concatenating
df3 = df3.drop(columns=['gender', 'is_deceased'], errors='ignore')

# 5. Vertically stack them securely
df_combined = pd.concat([df3, df4], axis=0, ignore_index=True)

In [11]:
import pandas as pd

# --- [Your predefined m3_exclusive and m4_exclusive lists go here] ---
m3_exclusive = [
    'report_10', 'report_100', 'report_11', 'report_12', 'report_13', 'report_14', 'report_15',
    'report_18', 'report_20', 'report_24', 'report_25', 'report_30', 'report_admission', 'report_artery',
    'report_blood', 'report_chest', 'report_ct', 'report_daily', 'report_daily_daily', 'report_date',
    'report_day', 'report_discharge', 'report_disease', 'report_dr', 'report_given', 'report_history',
    'report_home', 'report_hospital', 'report_hospital1', 'report_hours', 'report_medications', 'report_mg',
    'report_mg_tablet', 'report_normal', 'report_pain', 'report_patient', 'report_po', 'report_po_daily',
    'report_pt', 'report_right', 'report_sig', 'report_sig_tablet', 'report_status', 'report_tablet',
    'report_tablet_po', 'report_tablet_sig', 'report_time', 'report_times', 'report_valve', 'title_acute',
    'title_atherosclerosis', 'title_atherosclerosis_native', 'title_care', 'title_cerebral', 'title_chronic',
    'title_coronary_atherosclerosis', 'title_episode', 'title_episode_care', 'title_failure', 'title_heart_failure',
    'title_hemorrhage', 'title_infarction', 'title_infarction_initial', 'title_initial', 'title_initial_episode',
    'title_malignant', 'title_mention', 'title_native', 'title_native_coronary', 'title_septicemia',
    'title_subendocardial', 'title_subendocardial_infarction', 'title_unspecified', 'title_unspecified_septicemia',
    'title_valve'
]

m4_exclusive = [
    'report_1st', 'report_1st_degree', 'report_abnormal', 'report_abnormal_ecg', 'report_age',
    'report_age_undetermined', 'report_anterior', 'report_atrial', 'report_atrial_fibrillation', 'report_axis',
    'report_axis_deviation', 'report_block', 'report_borderline', 'report_changes', 'report_changes_myocardial',
    'report_changes_nonspecific', 'report_degree', 'report_degree_block', 'report_deviation', 'report_ecg',
    'report_ecg_sinus', 'report_extensive', 'report_fibrillation', 'report_hypertrophy', 'report_infarct',
    'report_infarct_age', 'report_inferior', 'report_inferior_infarct', 'report_inferior_lateral', 'report_ischemia',
    'report_lateral', 'report_lateral_st', 'report_leads', 'report_myocardial', 'report_myocardial_ischemia',
    'report_nonspecific', 'report_possible', 'report_response', 'report_rhythm', 'report_sinus', 'report_sinus_rhythm',
    'report_st', 'report_st_changes', 'report_undetermined', 'report_ventricular', 'report_ventricular_hypertrophy',
    'report_ventricular_response', 'report_wave', 'report_wave_changes', 'title_angioplasty', 'title_approach',
    'title_arteriography', 'title_arteriography_using', 'title_cardiac', 'title_cardiac_catheterization',
    'title_catheterization', 'title_catheters', 'title_coronary_arteriography', 'title_drug', 'title_drug_eluting',
    'title_eluting', 'title_heart_cardiac', 'title_insertion', 'title_left', 'title_left_heart', 'title_percutaneous',
    'title_percutaneous_approach', 'title_procedure', 'title_right', 'title_single', 'title_stent',
    'title_transluminal', 'title_transluminal_coronary', 'title_using', 'title_using_catheters'
]
# 2. Add missing MIMIC-IV columns to df3 with 0
for col in m4_exclusive:
    if col not in df3.columns:
        df3[col] = 0

# 3. FIX: Add missing MIMIC-III columns to df4 with 0
for col in m3_exclusive:  # Fixed: changed from m4_exclusive to m3_exclusive
    if col not in df4.columns:
        df4[col] = 0

# 4. Standardize demographics before concatenating
df3 = df3.drop(columns=['gender', 'is_deceased'], errors='ignore')

# Ensure columns are in the exact same order before concat
df4 = df4[df3.columns]

# 5. Vertically stack them securely
df_combined = pd.concat([df3, df4], axis=0, ignore_index=True)
print("Combined shape:", df_combined.shape)

Combined shape: (54151, 168)


In [13]:
# Check which columns contain missing values
nan_counts = X.isna().sum()
print("Columns with missing values:")
print(nan_counts[nan_counts > 0])

NameError: name 'X' is not defined

In [12]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# =============================================================
# FIXED: Handle Missing Values (Imputation) & Feature Scaling
# =============================================================

# 1. Initialize the Median Imputer
imputer = SimpleImputer(strategy='median')

# Fit on training data and transform both sets
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 2. Scale the imputed data (Crucial for SAGA/ElasticNet)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

NameError: name 'X_train' is not defined

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# 1. Define Features, Target, and Groups
# ==========================================
# Target column
target_col = 'hospital_expire_flag'

# Grouping by patient to avoid data leakage during cross-validation
groups = df_combined['subject_id']

# Drop identifiers and labels to isolate features
X = df_combined.drop(columns=['subject_id', 'hadm_id', 'icd_code', 'icd_description', target_col], errors='ignore')
y = df_combined[target_col]

# Split into Train/Test using GroupKFold to keep patient histories unmixed
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# =============================================================
# FIX: Impute Missing Values & Scale (Crucial for SAGA/ElasticNet)
# =============================================================
# 1. Initialize the Median Imputer to handle any missing EHR vital signs safely
imputer = SimpleImputer(strategy='median')

# Fit on training data and transform both sets
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# 2. Scale the imputed data cleanly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# ==========================================
# 3. Hyperparameter Search Space Setup
# # ==========================================
# param_distributions = {
#     'C': np.logspace(-2, 1, 10),
#     'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
# }

# cv_strategy = GroupKFold(n_splits=5)

# base_lr = LogisticRegression(
#     penalty='elasticnet',
#     solver='saga',
#     class_weight='balanced',
#     max_iter=1000, # Kept high to ensure convergence over combined text tokens
#     random_state=42
# )

# print("Running fast RandomizedSearchCV over ElasticNet parameters...")
# lr_search = RandomizedSearchCV(
#     estimator=base_lr,
#     param_distributions=param_distributions,
#     n_iter=8,
#     scoring='roc_auc',
#     cv=cv_strategy.split(X_train_scaled, y_train, groups=groups_train),
#     random_state=42,
#     n_jobs=-1
# )

# # This will now run smoothly without any NaN errors!
# lr_search.fit(X_train_scaled, y_train)

# ==========================================
# 3. FAST Hyperparameter Search Space Setup
# ==========================================
# We focus strictly on the regularization strength 'C'
param_distributions = {
    'C': np.logspace(-3, 2, 10)
}

cv_strategy = GroupKFold(n_splits=5)

# Swapping to liblinear + L2 for near-instant execution on sparse text data
base_lr = LogisticRegression(
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=500, # liblinear converges very rapidly
    random_state=42
)

print("Running ultra-fast RandomizedSearchCV using liblinear solver...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=6, # Evaluates 6 random candidates across the 5 splits
    scoring='roc_auc',
    cv=cv_strategy.split(X_train_scaled, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

# This will complete in seconds
# This will complete in seconds
lr_search.fit(X_train_scaled, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f}")

# REMOVE OR COMMENT OUT THIS DUPLICATE LINE THAT CAUSES THE ERROR:
# print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['lr_ratio']:.2f}")

# ==========================================
# 4. Evaluate Tuned Estimator on Holdout Test Split
# ==========================================
best_model = lr_search.best_estimator_

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f}")

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
# print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['lr_ratio']:.2f}")

# ==========================================
# 4. Evaluate Tuned Estimator on Holdout Test Split
# ==========================================
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test_scaled)[:, 1]

# 5. Build Evaluation Summary Table Linked by Patient Keys
test_summary = pd.DataFrame({
    'subject_id': df_combined.iloc[test_idx]['subject_id'].values,
    'y_true': y_test.values,
    'y_prob': y_prob_tuned
})

# 6. Collapse down to Patient-Level Summary Metrics safely
patient_res_true = test_summary.groupby('subject_id')['y_true'].max()
patient_res_pred = test_summary.groupby('subject_id')['y_prob'].mean()

# Binary classification cutoff threshold
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Optimized Cohort Patient-Level Evaluation ---")
print(classification_report(patient_res_true, preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Running ultra-fast RandomizedSearchCV using liblinear solver...

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================
Best Parameters Found -> C: 0.0036

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================
Best Parameters Found -> C: 0.0036

================ OPTIMIZATION SEARCH RESULTS COMPLETE ================

--- Optimized Cohort Patient-Level Evaluation ---
              precision    recall  f1-score   support

         0.0       0.96      0.88      0.92      5307
         1.0       0.55      0.83      0.66       984

    accuracy                           0.87      6291
   macro avg       0.76      0.85      0.79      6291
weighted avg       0.90      0.87      0.88      6291

Optimized Patient-Level ROC-AUC Score: 0.9376


In [20]:
# this is for combined both mimic3 and 4

In [ ]:
# n

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

# ==========================================
# NEW: Define Features, Target, and Groups
# ==========================================
# Target column (adjust name if it differs in your dataframe)
target_col = 'hospital_expire_flag'

# Grouping by patient to avoid data leakage during cross-validation
groups = df_combined['subject_id']

# Drop identifiers and labels to isolate features
X = df_combined.drop(columns=['subject_id', 'hadm_id', 'icd_code', 'icd_description', target_col], errors='ignore')
y = df_combined[target_col]

# Split into Train/Test using GroupKFold to keep patient histories unmixed
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# ==========================================
# NEW: Feature Scaling (Crucial for SAGA/ElasticNet)
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 1. Hyperparameter Search Space Setup
# ==========================================
param_distributions = {
    'C': np.logspace(-2, 1, 10),
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

cv_strategy = GroupKFold(n_splits=5)

base_lr = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=1000, # Increased to guarantee convergence with sparse text variables
    random_state=42
)

print("Running fast RandomizedSearchCV over ElasticNet parameters...")
lr_search = RandomizedSearchCV(
    estimator=base_lr,
    param_distributions=param_distributions,
    n_iter=8,
    scoring='roc_auc',
    cv=cv_strategy.split(X_train_scaled, y_train, groups=groups_train),
    random_state=42,
    n_jobs=-1
)

lr_search.fit(X_train_scaled, y_train)

print("\n================ OPTIMIZATION SEARCH RESULTS COMPLETE ================")
print(f"Best Parameters Found -> C: {lr_search.best_params_['C']:.4f} | l1_ratio: {lr_search.best_params_['l1_ratio']:.2f}")

# ==========================================
# 2. Evaluate Tuned Estimator on Holdout Test Split
# ==========================================
best_model = lr_search.best_estimator_
y_prob_tuned = best_model.predict_proba(X_test_scaled)[:, 1]

# 3. Build Evaluation Summary Table Linked by Patient Keys
test_summary = pd.DataFrame({
    'subject_id': df_combined.iloc[test_idx]['subject_id'].values, # FIX: use df_combined instead of undefined 'df'
    'y_true': y_test.values,
    'y_prob': y_prob_tuned
})

# 4. Collapse down to Patient-Level Summary Metrics safely
patient_res_true = test_summary.groupby('subject_id')['y_true'].max()
patient_res_pred = test_summary.groupby('subject_id')['y_prob'].mean()

# Binary classification cutoff threshold
preds = (patient_res_pred >= 0.5).astype(int)

print("\n--- Optimized Cohort Patient-Level Evaluation ---")
print(classification_report(patient_res_true, preds))
print(f"Optimized Patient-Level ROC-AUC Score: {roc_auc_score(patient_res_true, patient_res_pred):.4f}")

Running fast RandomizedSearchCV over ElasticNet parameters...


ValueError: 
All the 40 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1222, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py", line 2961, in validate_data
    X, y = check_X_y(X, y, **che